# Lab Regularisasi: Standardisasi, Lasso, dan Ridge

Course 2: Supervised Learning

Tujuan latihan ini:

- melakukan standardisasi data dan melihat efeknya pada koefisien
- mencoba Lasso dan Ridge, serta membandingkan keduanya
- menggabungkan standardisasi dengan train-test split tanpa kebocoran data
- melihat bagaimana regularisasi mencegah overfitting

## Dataset

Datanya `boston_housing_clean.pickle` (dictionary berisi `dataframe` dan `description`), target `MEDV`:

https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/data/boston_housing_clean.pickle

## 1. Import dan baca data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Lasso, Ridge
from sklearn.metrics import r2_score

np.set_printoptions(precision=3, suppress=True)

In [ ]:
URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-ML240EN-SkillsNetwork/labs/data/boston_housing_clean.pickle"
boston = pd.read_pickle(URL)
df = boston["dataframe"]

X = df.drop(columns="MEDV")
y = df["MEDV"]
print(X.shape)
df.head()

## 2. Standardisasi

Standardisasi mengubah tiap fitur supaya rata-ratanya 0 dan standar deviasinya 1: $z = (x - \mu) / \sigma$. Di scikit-learn dilakukan dengan `StandardScaler`.

In [ ]:
scaler = StandardScaler()
X_ss = scaler.fit_transform(X)
print("rata-rata tiap kolom (harusnya ~0):", X_ss.mean(axis=0).round(6)[:5], "...")
print("std tiap kolom (harusnya 1)       :", X_ss.std(axis=0).round(6)[:5], "...")

### Latihan: buktikan dengan hitungan manual

In [ ]:
X_arr = X.to_numpy()
manual = (X_arr - X_arr.mean(axis=0)) / X_arr.std(axis=0)
print("hasil manual sama dengan StandardScaler:", np.allclose(manual, X_ss))

## 3. Koefisien dengan dan tanpa standardisasi

In [ ]:
lr_asli = LinearRegression().fit(X, y)
lr_std = LinearRegression().fit(X_ss, y)

koef = pd.DataFrame({"tanpa scaling": lr_asli.coef_, "dengan standardisasi": lr_std.coef_}, index=X.columns)
koef.round(3)

Diskusi: koefisien tanpa scaling berada di skala yang sangat berbeda antar fitur, misalnya fitur dengan satuan kecil punya koefisien besar. Itu belum tentu "buruk" untuk prediksi regresi linear biasa, tapi koefisiennya jadi tidak bisa dibandingkan antar fitur. Setelah standardisasi semua fitur berada pada skala yang sama, jadi koefisien bisa dibandingkan.

### Latihan: fitur mana yang paling berpengaruh?

In [ ]:
koef["dengan standardisasi"].abs().sort_values(ascending=False).head(5).to_frame("|koefisien|") \
    .join(koef["dengan standardisasi"].rename("koefisien (bertanda)"))

Lima baris teratas di tabel itu adalah fitur yang paling berpengaruh. Tanda koefisien menunjukkan arahnya: positif berarti makin besar fitur, harga naik; negatif berarti harga turun. Jawabannya agak subjektif karena scikit-learn tidak memberi uji signifikansi statistik, jadi ini hanya berdasarkan besar koefisien.

## 4. Lasso dengan dan tanpa scaling

Pertanyaan:

1. Perbedaan Lasso dengan regresi linear biasa: Lasso menambah penalti L1 (jumlah nilai absolut koefisien) pada fungsi yang diminimalkan, sehingga koefisien dikecilkan dan sebagian bisa menjadi tepat nol.
2. Standardisasi lebih penting untuk Lasso, karena besar penalti bergantung pada besar koefisien. Fitur dengan skala kecil butuh koefisien besar, sehingga lebih "dihukum" daripada fitur berskala besar, padahal belum tentu kurang penting.

Aku buat fitur polinomial derajat 2 supaya ada banyak fitur untuk disaring. `include_bias=False` karena Lasso sudah punya intercept sendiri.

In [ ]:
pf = PolynomialFeatures(degree=2, include_bias=False)
X_pf = pf.fit_transform(X)
print(X.shape, "->", X_pf.shape)

X_pf_s = StandardScaler().fit_transform(X_pf)

### Latihan: alpha 0.1 vs 1

In [ ]:
def ringkas_lasso(alpha):
    m = Lasso(alpha=alpha, max_iter=100000).fit(X_pf_s, y)
    return {
        "alpha": alpha,
        "jumlah |koefisien|": np.abs(m.coef_).sum(),
        "koefisien tidak nol": int((m.coef_ != 0).sum()),
        "R2 (data yang sama)": r2_score(y, m.predict(X_pf_s)),
    }

pd.DataFrame([ringkas_lasso(0.1), ringkas_lasso(1)]).round(3)

Pertanyaan: apakah regularisasi bisa menaikkan performa kalau dievaluasi pada data yang sama dengan data latih? Tidak. Penalti justru membuat model sedikit kurang cocok dengan data latih, jadi $R^2$ pada data latih akan turun atau sama saja. Manfaat regularisasi baru terlihat di data yang belum pernah dilihat model.

## 5. Dengan train-test split

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X_pf, y, test_size=0.3, random_state=72018)

sc = StandardScaler().fit(X_tr)
X_tr_s = sc.transform(X_tr)
X_te_s = sc.transform(X_te)
print(X_tr_s.shape, X_te_s.shape)

In [ ]:
def evaluasi(model):
    model.fit(X_tr_s, y_tr)
    return {
        "R2 test": r2_score(y_te, model.predict(X_te_s)),
        "jumlah |koefisien|": np.abs(model.coef_).sum(),
        "koefisien tidak nol": int((model.coef_ != 0).sum()),
    }

hasil = pd.DataFrame({
    "Lasso alpha=1": evaluasi(Lasso(alpha=1, max_iter=100000)),
    "Lasso alpha=0.1": evaluasi(Lasso(alpha=0.1, max_iter=100000)),
    "Lasso alpha=0.001": evaluasi(Lasso(alpha=0.001, max_iter=100000)),
    "Regresi linear": evaluasi(LinearRegression()),
}).T
hasil.round(3)

Yang perlu dibandingkan dari tabel itu:

- Lasso dengan alpha 0.001 (dengan `max_iter=100000` supaya konvergen) vs regresi linear biasa pada $R^2$ test.
- Jumlah |koefisien| dan banyaknya koefisien nol. Model dengan jumlah koefisien lebih kecil dan lebih banyak nol adalah deskripsi hubungan yang lebih sederhana.

## 6. L1 vs L2: Lasso dan Ridge

Lasso dan Ridge dipakai dengan sintaks yang sama di scikit-learn. Aku bandingkan keduanya dengan `alpha` yang sama (0.001):

In [ ]:
ridge = Ridge(alpha=0.001).fit(X_tr_s, y_tr)
lasso = Lasso(alpha=0.001, max_iter=100000).fit(X_tr_s, y_tr)

pd.DataFrame({
    "jumlah |koefisien|": [np.abs(ridge.coef_).sum(), np.abs(lasso.coef_).sum()],
    "koefisien tidak nol": [int((ridge.coef_ != 0).sum()), int((lasso.coef_ != 0).sum())],
    "R2 test": [r2_score(y_te, ridge.predict(X_te_s)), r2_score(y_te, lasso.predict(X_te_s))],
    "R2 train": [r2_score(y_tr, ridge.predict(X_tr_s)), r2_score(y_tr, lasso.predict(X_tr_s))],
}, index=["Ridge", "Lasso"]).round(3)

Ridge tidak pernah membuat koefisien tepat nol, sedangkan Lasso bisa. Mana yang punya R2 lebih tinggi dan seberapa kuat regularisasinya pada alpha yang sama bergantung pada datanya, jadi tidak bisa digeneralisasi.

## 7. Apakah urutan scaling terhadap split berpengaruh?

Untuk regresi linear biasa, bandingkan dua cara: scaling seluruh data lalu split, versus split lalu scaling pakai statistik data train.

In [ ]:
# cara 1: scaling seluruh data dulu, baru split
a_tr, a_te, b_tr, b_te = train_test_split(X_ss, y, test_size=0.3, random_state=72018)
pred_1 = LinearRegression().fit(a_tr, b_tr).predict(a_te)

# cara 2: split dulu, scaler hanya belajar dari data train
c_tr, c_te, d_tr, d_te = train_test_split(X, y, test_size=0.3, random_state=72018)
sc2 = StandardScaler().fit(c_tr)
pred_2 = LinearRegression().fit(sc2.transform(c_tr), d_tr).predict(sc2.transform(c_te))

print("R2 cara 1:", round(r2_score(b_te, pred_1), 6))
print("R2 cara 2:", round(r2_score(d_te, pred_2), 6))
print("selisih prediksi terbesar:", np.abs(pred_1 - pred_2).max())

Untuk regresi linear biasa, kedua cara menghasilkan prediksi yang praktis identik, karena model ini tidak terpengaruh perubahan skala linear pada fitur. Tapi untuk model lain seperti Lasso, Ridge, atau SGD, urutannya berpengaruh, dan cara yang benar tetap yang kedua (split dulu). Nanti hal ini bisa dirapikan dengan memasukkan scaler ke dalam `Pipeline`.